In [0]:
%sql
select * from workspace.sliver.cust_info

customer_id,customer_key,customer_firstname,customer_lastname,customer_marital_status,customer_gender,customer_create_date
11000,AW00011000,Jon,Yang,MARRIED,MALE,2025-10-06
11001,AW00011001,Eugene,Huang,N/A,MALE,2025-10-06
11002,AW00011002,Ruben,Torres,MARRIED,MALE,2025-10-06
11003,AW00011003,Christy,Zhu,N/A,FEMALE,2025-10-06
11004,AW00011004,Elizabeth,Johnson,N/A,FEMALE,2025-10-06
11005,AW00011005,Julio,Ruiz,N/A,MALE,2025-10-06
11006,AW00011006,Janet,Alvarez,N/A,FEMALE,2025-10-06
11007,AW00011007,Marco,Mehta,MARRIED,MALE,2025-10-06
11008,AW00011008,Rob,Verhoff,N/A,FEMALE,2025-10-06
11009,AW00011009,Shannon,Carlson,N/A,MALE,2025-10-06


In [0]:
%sql
select * from workspace.sliver.sales_loc_a101

CID,COUNTRY
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia
AW00011005,Australia
AW00011006,Australia
AW00011007,Australia
AW00011008,Australia
AW00011009,Australia


In [0]:
%sql
select * from workspace.sliver.cust_az12

BDATE,GEN,cid_prefix,cid_rest
1971-10-06,Male,NAS,AW00011000
1976-05-10,Male,NAS,AW00011001
1971-02-09,Male,NAS,AW00011002
1973-08-14,Female,NAS,AW00011003
1979-08-05,Female,NAS,AW00011004
1976-08-01,Male,NAS,AW00011005
1976-12-02,Female,NAS,AW00011006
1969-11-06,Male,NAS,AW00011007
1975-07-04,Female,NAS,AW00011008
1969-09-29,Male,NAS,AW00011009


In [0]:

query = """

 select 
        row_number() over(order by customer_id) as customer_key,
        ci.customer_id,
        ci.customer_firstname as firstname,
        ci.customer_lastname as lastname,
        ci.customer_marital_status as marital_status,
        --ci.customer_gender as gender,
        ci.customer_create_date as creation_date,
        az.BDATE as brith_date,
        --az.GEN,
        lo.country,
        case when ci.customer_gender = 'N/A' then az.GEN
             when ci.customer_gender != 'N/A' then ci.customer_gender
             else ci.customer_gender
        end as gender
    from workspace.sliver.cust_info as ci
    left join workspace.sliver.cust_az12 as az
        on ci.customer_key = az.cid_rest
    left join workspace.sliver.sales_loc_a101 as lo
        on ci.customer_key = lo.CID;
    """
df = spark.sql(query)
df.write.mode("overwrite").saveAsTable("gold.cust_info")

In [0]:
df.createOrReplaceTempView("tmp_dim_customers")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_customers
USING DELTA
AS
SELECT * FROM tmp_dim_customers;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.gold.dim_customers

customer_key,customer_id,firstname,lastname,marital_status,creation_date,brith_date,country,gender
1,null,null,null,N/A,null,null,null,null
2,null,null,null,N/A,null,null,null,null
3,null,null,null,N/A,null,null,null,null
4,null,null,null,N/A,null,null,null,null
5,11000,Jon,Yang,MARRIED,2025-10-06,1971-10-06,Australia,MALE
6,11001,Eugene,Huang,N/A,2025-10-06,1976-05-10,Australia,MALE
7,11002,Ruben,Torres,MARRIED,2025-10-06,1971-02-09,Australia,MALE
8,11003,Christy,Zhu,N/A,2025-10-06,1973-08-14,Australia,FEMALE
9,11004,Elizabeth,Johnson,N/A,2025-10-06,1979-08-05,Australia,FEMALE
10,11005,Julio,Ruiz,N/A,2025-10-06,1976-08-01,Australia,MALE
